# Data contoh

<a href="https://colab.research.google.com/github/aldo02032004/sentiment-volume-forecast/blob/main/examples/make_sample_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Bikin data palsu dengan format export yang sama (`sample_export.xlsx`). File ini yang dipakai sebagai data contoh di notebook utama.

# 1. Library

In [1]:
import numpy as np    # angka acak
import pandas as pd   # tabel data

print("Library siap")

Library siap


# 2. Pengaturan

In [2]:
DAYS = 120             # panjang data (hari)
START = "2026-05-01"   # tanggal mulai
SEED = 7               # ganti angka ini buat dapat data contoh lain
SENTIMENTS = {"positive": (90, 0.2), "negative": (120, -0.1), "neutral": (50, 0.0)}   # (rata-rata dasar, tren per hari)

rng = np.random.default_rng(SEED)
dates = pd.date_range(START, periods=DAYS, freq="D")
print(f"{dates[0]:%d %b %Y} - {dates[-1]:%d %b %Y} ({DAYS} hari)")

01 May 2026 - 28 Aug 2026 (120 hari)


# 3. Rata-rata harian per sentimen

In [3]:
day = np.arange(DAYS)
level = {}
for s, (base, trend) in SENTIMENTS.items():
    level[s] = base + trend * day - 20 * (dates.dayofweek >= 5) + 15 * np.sin(day / 9)   # tren + akhir pekan sepi + gelombang
    level[s] *= np.where(rng.random(DAYS) < 0.05, rng.uniform(2, 3.5, DAYS), 1)          # sesekali lonjakan isu (±5% hari)
level = pd.DataFrame(level, index=dates).clip(lower=5)
level.round(0).head(10)

,positive,negative,neutral
2026-05-01,90.0,120.0,50.0
2026-05-02,72.0,102.0,32.0
2026-05-03,74.0,103.0,33.0
2026-05-04,96.0,125.0,55.0
2026-05-05,97.0,126.0,56.0
2026-05-06,99.0,127.0,58.0
2026-05-07,231.0,129.0,59.0
2026-05-08,102.0,130.0,61.0
2026-05-09,83.0,111.0,42.0
2026-05-10,84.0,112.0,43.0


# 4. Jumlah postingan per hari

In [4]:
counts = pd.DataFrame(rng.poisson(level), index=dates, columns=level.columns)   # angka acak di sekitar rata-rata harian
print(f"Total {counts.to_numpy().sum():,} postingan")
counts.head(10)

Total 32,380 postingan


,positive,negative,neutral
2026-05-01,76,139,44
2026-05-02,51,91,41
2026-05-03,64,104,32
2026-05-04,103,136,61
2026-05-05,110,125,57
2026-05-06,86,103,48
2026-05-07,224,121,55
2026-05-08,94,139,56
2026-05-09,76,109,54
2026-05-10,90,118,53


# 5. Satu baris per postingan

In [5]:
df = counts.stack().rename_axis(["Date", "Sentiment"]).reset_index(name="n")
df = df.loc[df.index.repeat(df["n"])].drop(columns="n").reset_index(drop=True)   # 1 baris = 1 postingan
df["Date"] += pd.to_timedelta(rng.integers(0, 86400, len(df)), unit="s")          # jam acak dalam sehari
df["Author"] = [f"user_{a}" for a in rng.integers(0, 3000, len(df))]               # 3.000 akun
df = df.sort_values("Date", ignore_index=True)
df.head()

,Date,Sentiment,Author
0,2026-05-01 00:00:53,neutral,user_1524
1,2026-05-01 00:03:51,positive,user_2287
2,2026-05-01 00:08:13,positive,user_2863
3,2026-05-01 00:17:13,positive,user_2829
4,2026-05-01 00:17:52,negative,user_2916


# 6. Kolom lain ala file export

In [6]:
n = len(df)
df = df.assign(No=np.arange(1, n + 1), Type="Tweet", Headline="", Mentions="contoh postingan",
               Link="https://example.com", Media="Twitter", Followers=rng.integers(10, 50000, n),
               Retweeted=rng.integers(0, 100, n), Favourited=rng.integers(0, 300, n), Location="",
               Date=df["Date"].dt.strftime("%Y-%m-%d %H:%M:%S"))
df = df[["No", "Type", "Headline", "Mentions", "Date", "Link", "Media", "Sentiment",
         "Author", "Followers", "Retweeted", "Favourited", "Location"]]
df.head()

,No,Type,Headline,Mentions,Date,Link,Media,Sentiment,Author,Followers,Retweeted,Favourited,Location
0,1,Tweet,,contoh postingan,2026-05-01 00:00:53,https://example.com,Twitter,neutral,user_1524,24954,79,158,
1,2,Tweet,,contoh postingan,2026-05-01 00:03:51,https://example.com,Twitter,positive,user_2287,31555,27,22,
2,3,Tweet,,contoh postingan,2026-05-01 00:08:13,https://example.com,Twitter,positive,user_2863,9350,7,5,
3,4,Tweet,,contoh postingan,2026-05-01 00:17:13,https://example.com,Twitter,positive,user_2829,15001,1,64,
4,5,Tweet,,contoh postingan,2026-05-01 00:17:52,https://example.com,Twitter,negative,user_2916,48804,70,137,


# 7. Simpan

In [7]:
df.to_excel("sample_export.xlsx", index=False, startrow=1)   # baris 1 kosong, header di baris 2 (sama kayak export asli)
print(f"sample_export.xlsx: {len(df):,} baris")

sample_export.xlsx: 32,380 baris
